<a href="https://colab.research.google.com/github/fujisawatet/Spotify_data/blob/main/%E3%82%B8%E3%83%A3%E3%83%B3%E3%83%AB%E5%88%A5_2020_2021_SHAP%E3%82%AF%E3%83%A9%E3%82%B9%E3%82%BF%E3%83%AA%E3%83%B3%E3%82%B0_k_pop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#7.21 必要なコードだけに整理

**k-pop 版**: hip-hop 版（`..._hip_hop.ipynb`）をコピーし、`TARGET_GENRE = 'k-pop'` に変えたもの。Step9・Step10 は k-pop 版のみ。

In [ ]:
#@title import
from urllib.request import urlretrieve
import os
from zipfile import ZipFile

import pandas as pd

In [ ]:
boxurl = "https://tus.box.com/shared/static/491ie3a5kdgg7hfajivmwlou6zq2fnzj.zip"
filename = "Kaggle_Spotify_1Million_Tracks.zip"

In [ ]:
%%time
urlretrieve(boxurl, filename)

In [ ]:
ZipFile(filename).extractall()

In [ ]:
%pip install japanize-matplotlib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import japanize_matplotlib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

print(f"shap バージョン: {shap.__version__}")

In [ ]:
df = pd.read_csv("/content/spotify_data.csv", index_col=0)
print(f"データ形状: {df.shape}")

df_clean = df.dropna(subset=['popularity']).copy()

# Step1 ;2020年単体データ絞り込み・SHAP値計算

In [ ]:
# genre_enc を除いた15特徴量 + gb2 モデルの再定義
FEATURE_COLS_V2 = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_ms', 'key', 'mode', 'time_signature',
    #'artist_freq'
]

print("HistGradientBoosting (no genre_enc) training...")
gb2 = HistGradientBoostingRegressor(
        max_iter=100, learning_rate=0.05, max_depth=3,
        min_samples_leaf=50, l2_regularization=1.0, random_state=42
    )
#HistGradientBoostingRegressor(
#    max_iter=300, learning_rate=0.05, max_depth=8,
#    min_samples_leaf=20, l2_regularization=0.1, random_state=42
#)

In [ ]:
df_clean['genre'].value_counts().sort_values(ascending=False).head(15)

In [ ]:
#@title filtering 2020/genre sampling
TARGET_YEAR = 2020
TARGET_GENRE = 'k-pop'#@param['black-metal','hip-hop','k-pop']
TEST_YEAR = 2021
df_2020 = df_clean[df_clean['year'] == TARGET_YEAR].copy()
df_2021 = df_clean[df_clean['year'] == TEST_YEAR].copy()
df_2020 = df_2020[df_2020['genre']==TARGET_GENRE]
df_2021 = df_2021[df_2021['genre']==TARGET_GENRE]

print(f"filter: year == {TARGET_YEAR}")
print(f"records: {len(df_2020):,} / {len(df_2020)/len(df_clean)*100:.1f}% of all data")
print(f"popularity: mean={df_2020['popularity'].mean():.1f}, "
      f"median={df_2020['popularity'].median():.1f}, "
      f"std={df_2020['popularity'].std():.2f}")
print(f"\ntop 10 genres:")
print(df_2020['genre'].value_counts().head(10).to_string())

N_MAX3 = 10000
rng4 = np.random.default_rng(42)
if len(df_2020) > N_MAX3:
    idx_a = rng4.choice(len(df_2020), size=N_MAX3, replace=False)
    df_2020_sample = df_2020.iloc[idx_a].reset_index(drop=True)
    print(f"\nsampled: {len(df_2020):,} -> {N_MAX3:,}")
else:
    df_2020_sample = df_2020.reset_index(drop=True)
    print(f"\nall records used: {len(df_2020_sample):,}")

X_2020_sample = df_2020_sample[FEATURE_COLS_V2]


In [ ]:
X2_train = df_2020[FEATURE_COLS_V2]
X2_test  = df_2021[FEATURE_COLS_V2]
y_train = df_2020['popularity']
y_test = df_2021['popularity']

In [ ]:
%%time
gb2.fit(X2_train, y_train)
gb2_pred = gb2.predict(X2_test)
print(f"done / MAE={mean_absolute_error(y_test, gb2_pred):.4f}, R2={r2_score(y_test, gb2_pred):.4f}")


In [ ]:
%%time
print("building SHAP Explainer (no genre_enc)...")
rng = np.random.default_rng(42)
# 背景データは最大100件（hip-hop 2020年は854曲しかなく900件は取れない。SHAPは内部で100件に間引くので結果はほぼ同じ）
bg_idx = rng.choice(len(X2_train), size=min(100, len(X2_train)), replace=False)
X2_bg = X2_train[FEATURE_COLS_V2].iloc[bg_idx]
explainer_gb2 = shap.Explainer(gb2, X2_bg)
print("done")

# 確認: 基準値が学習データの人気度平均に近ければ、今のジャンルのモデルで作られている
base_val = float(np.ravel(explainer_gb2.expected_value)[0])
print(f"expected_value = {base_val:.2f} / {TARGET_GENRE} {TARGET_YEAR} popularity mean = {y_train.mean():.2f}")

In [ ]:
%%time
#time 19sec
print("computing SHAP values...")
shap_values_v3 = explainer_gb2(X2_test, check_additivity=False)
shap_matrix_v3 = shap_values_v3.values
print(f"done / shap_matrix_v3 shape: {shap_matrix_v3.shape}")

#21年でのテスト評価

#SHAP Plots

In [ ]:
# shap_values_v3 はこの時点で2021年の曲のSHAP値なので、特徴量も2021年（X2_test）を渡す
shap.summary_plot(shap_values_v3, X2_test, plot_type="bar", show=False)

In [ ]:
shap.initjs()
shap.force_plot(explainer_gb2.expected_value, shap_values_v3.values[0], X2_test.iloc[0])

#Step 2: クラスタ数の探索

In [ ]:
#18sec
%%time
TARGET_YEAR = 2020
#df_2020 = df_clean[df_clean['year'] == TARGET_YEAR].copy()

print(f"filter: year == {TARGET_YEAR}")
print(f"records: {len(df_2020):,} / {len(df_2020)/len(df_clean)*100:.1f}% of all data")
print(f"popularity: mean={df_2020['popularity'].mean():.1f}, "
      f"median={df_2020['popularity'].median():.1f}, "
      f"std={df_2020['popularity'].std():.2f}")
print(f"\ntop 10 genres:")
print(df_2020['genre'].value_counts().head(10).to_string())

N_MAX3 = 10000
rng4 = np.random.default_rng(42)
if len(df_2020) > N_MAX3:
    idx_a = rng4.choice(len(df_2020), size=N_MAX3, replace=False)
    df_2020_sample = df_2020.iloc[idx_a].reset_index(drop=True)
    print(f"\nsampled: {len(df_2020):,} -> {N_MAX3:,}")
else:
    df_2020_sample = df_2020.reset_index(drop=True)
    print(f"\nall records used: {len(df_2020_sample):,}")

X_2020_sample = df_2020_sample[FEATURE_COLS_V2]

print("computing SHAP values...")
shap_values_v3 = explainer_gb2(X_2020_sample, check_additivity=False)
shap_matrix_v3 = shap_values_v3.values
print(f"done / shap_matrix_v3 shape: {shap_matrix_v3.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler_v3 = StandardScaler()
shap_scaled_v3 = scaler_v3.fit_transform(shap_matrix_v3)
print(f"standardized / shape: {shap_scaled_v3.shape}")

inertias_v3 = []
silhouettes_v3 = []
K_range_v3 = range(2, 11)

for k in K_range_v3:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(shap_scaled_v3)
    inertias_v3.append(km.inertia_)
    sil = silhouette_score(shap_scaled_v3, labels_k,
                           sample_size=min(2000, len(shap_scaled_v3)), random_state=42)
    silhouettes_v3.append(sil)
    print(f"  k={k}: inertia={km.inertia_:.1f}, silhouette={sil:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Step A2: k exploration (2020 only, no genre_enc)", fontsize=13, fontweight='bold')

axes[0].plot(list(K_range_v3), inertias_v3, 'b-o', linewidth=2)
axes[0].set_xlabel("k")
axes[0].set_ylabel("inertia")
axes[0].set_title("Elbow method")
axes[0].grid(alpha=0.3)

axes[1].plot(list(K_range_v3), silhouettes_v3, 'o-', color='darkorange', linewidth=2)
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette score")
axes[1].set_title("Silhouette score")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_2020_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

#Step3: クラスタリング実行（k=7）

In [ ]:
CHOSEN_K_V3 = 7  # silhouette score 最高値（0.126）

km_v3 = KMeans(n_clusters=CHOSEN_K_V3, random_state=42, n_init=10)
cluster_labels_v3 = km_v3.fit_predict(shap_scaled_v3)
df_2020['cluster_v3'] = cluster_labels_v3

chosen_sil_v3 = silhouettes_v3[CHOSEN_K_V3 - 2]
print(f"clustering done / k={CHOSEN_K_V3} (silhouette={chosen_sil_v3:.3f})")
print(f"\ncluster counts:")
print(df_2020['cluster_v3'].value_counts().sort_index().to_string())

print(f"\ncluster overview:")
for k in range(CHOSEN_K_V3):
    mask = df_2020['cluster_v3'] == k
    df_k = df_2020[mask]
    top3_g = df_k['genre'].value_counts().head(3).index.tolist()
    print(f"  cluster {k}: {len(df_k):,} tracks / pop mean={df_k['popularity'].mean():.1f} / {', '.join(top3_g)}")

In [ ]:
#@title scalingした値を代入
df_2020_sample = df_2020.copy()
df_2020_sample[FEATURE_COLS_V2] = shap_scaled_v3
df_2020_sample

In [ ]:
df_2020['cluster_v3'].unique()

In [ ]:
fig, axes = plt.subplots(2,4, figsize=(15,4))
for ax,(i,df) in zip(axes.ravel(), df_2020_sample.groupby('cluster_v3')[FEATURE_COLS_V2]):
  df[FEATURE_COLS_V2].boxplot(ax=ax)
  ax.set_title(f'cluster{i}')
plt.show()

#Step4: 箱ひげ図

In [ ]:
#@title plot features
n_cols = 3
n_rows = 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 25))
fig.suptitle("Step A4: boxplot / SHAP by cluster (2020, k=7)", fontsize=12, fontweight='bold')
colors = plt.cm.tab10(np.linspace(0, 1, CHOSEN_K_V3))
xlabels = [str(i) for i in range(CHOSEN_K_V3)]

for fi, fname in enumerate(FEATURE_COLS_V2):
    r = fi // n_cols
    c = fi % n_cols
    ax = axes[r, c]
    data = [shap_matrix_v3[cluster_labels_v3 == i, fi] for i in range(CHOSEN_K_V3)]
    bp = ax.boxplot(data, patch_artist=True, showfliers=False, medianprops=dict(color='black', linewidth=1.5))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    ax.set_title(fname, fontsize=10, fontweight='bold')
    ax.set_xlabel("cluster")
    ax.set_ylabel("SHAP")
    ax.set_xticks(range(1, CHOSEN_K_V3 + 1))
    ax.set_xticklabels(xlabels, fontsize=8)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)

for i in range(len(FEATURE_COLS_V2), n_rows * n_cols):
    axes[i // n_cols][i % n_cols].set_visible(False)

plt.tight_layout()
plt.savefig('phase3_2020_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved: phase3_2020_boxplot.png")

In [ ]:
#plot a box plot of all features for each cluster. each box plot should have all features in xticks and plot its value for one cluster
import matplotlib.pyplot as plt
import seaborn as sns
import math

n_clusters = df_2020_sample['cluster_v3'].nunique()
cols = 2
rows = math.ceil(n_clusters / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, 6 * rows))
axes = axes.ravel()

# Melt the dataframe so we can easily plot features on the x-axis
df_melted = df_2020_sample.melt(id_vars=['cluster_v3'], value_vars=FEATURE_COLS_V2, var_name='Feature', value_name='Value')

clusters = sorted(df_2020_sample['cluster_v3'].unique())
for i, cluster_id in enumerate(clusters):
    cluster_data = df_melted[df_melted['cluster_v3'] == cluster_id]
    sns.boxplot(x='Feature', y='Value', data=cluster_data, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Cluster {cluster_id} Features')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# まとめ出力（実行結果から自動で作る。ジャンルを変えてもそのまま使える）
print("=" * 70)
print(f"Summary: {TARGET_GENRE} / train {TARGET_YEAR} -> test {TEST_YEAR} / k={CHOSEN_K_V3}")
print("=" * 70)

print("\n[Data]")
print(f"  train {TARGET_YEAR}: {len(df_2020):,} tracks / test {TEST_YEAR}: {len(df_2021):,} tracks")
print(f"  popularity {TARGET_YEAR}: mean={df_2020['popularity'].mean():.1f}, std={df_2020['popularity'].std():.2f}")

print("\n[Model: gb2 (iter=100, depth=3, leaf=50, l2=1.0)]")
print(f"  test {TEST_YEAR}: MAE={mean_absolute_error(y_test, gb2_pred):.4f}, R2={r2_score(y_test, gb2_pred):.4f}")

print(f"\n[Clustering on SHAP values of {TARGET_YEAR} tracks]")
print(f"  k = {CHOSEN_K_V3} (silhouette = {chosen_sil_v3:.3f})")

labels_arr = df_2020['cluster_v3'].values
for k in range(CHOSEN_K_V3):
    d_k = df_2020[labels_arr == k]
    mean_shap = shap_matrix_v3[labels_arr == k].mean(axis=0)
    top_i = int(np.argmax(mean_shap))
    bot_i = int(np.argmin(mean_shap))
    line = f"  C{k}: {len(d_k):>4} tracks / pop={d_k['popularity'].mean():.1f}"
    line += f" / SHAP+ {FEATURE_COLS_V2[top_i]}({mean_shap[top_i]:+.2f})"
    line += f" / SHAP- {FEATURE_COLS_V2[bot_i]}({mean_shap[bot_i]:+.2f})"
    print(line)

print("\n" + "=" * 70)

#Step5: 各クラスタに分けられた曲のタイトル一覧

**打ち合わせメモ**: hip-hop の中で特徴によって違うから、タイトルなどを聞いて違いがわかるか / 各クラスタに分けられた曲のタイトルがわかるか

各クラスタについて次を表示する。

- **特徴量の傾向**: そのクラスタの曲が、ジャンル全体と比べてどの音楽的特徴が高い/低いか（標準化した値の平均）。聴くときに注目するポイントになる
- **代表曲**: クラスタの中心に最も近い曲。そのクラスタらしさが一番よく出ている曲
- **人気曲**: クラスタ内で popularity が高い曲。知っている曲が見つかりやすい
- **主なアーティスト**: 1人のアーティストの曲ばかりなら、クラスタは「音の違い」ではなく「アーティストの違い」を拾っている可能性がある



In [ ]:
# Step5: 各クラスタに分けられた曲のタイトル一覧
# 必要な変数: df_2020, shap_scaled_v3, km_v3, CHOSEN_K_V3, FEATURE_COLS_V2, TARGET_GENRE, TARGET_YEAR

N_SHOW = 5

df_tracks = df_2020.copy()
df_tracks['spotify_url'] = 'https://open.spotify.com/track/' + df_tracks['track_id'].astype(str)

# クラスタ中心からの距離（小さいほど、そのクラスタらしい曲）
labels_arr = df_tracks['cluster_v3'].values
centers = km_v3.cluster_centers_
df_tracks['dist_to_center'] = np.linalg.norm(shap_scaled_v3 - centers[labels_arr], axis=1)

# 音楽的特徴をジャンル内で標準化（0 = ジャンル平均、+1 = 平均より1標準偏差高い）
feat_z = (df_tracks[FEATURE_COLS_V2] - df_tracks[FEATURE_COLS_V2].mean()) / df_tracks[FEATURE_COLS_V2].std()


def print_track(row):
    title = str(row['track_name'])[:38].ljust(40)
    artist = str(row['artist_name'])[:22].ljust(24)
    pop = str(int(row['popularity'])).rjust(3)
    print("    " + title + artist + pop + "  " + row['spotify_url'])


for k in range(CHOSEN_K_V3):
    mask = labels_arr == k
    d_k = df_tracks[mask]

    print("=" * 110)
    print(f"cluster {k}: {len(d_k)} tracks / popularity mean = {d_k['popularity'].mean():.1f} / artists = {d_k['artist_name'].nunique()}")

    # 特徴量の傾向（ジャンル平均との差が大きい順に3つ）
    z_mean = feat_z[mask].mean().sort_values()
    high = z_mean.tail(3)[::-1]
    low = z_mean.head(3)
    print("  high: " + ", ".join(f"{f}({v:+.2f})" for f, v in high.items()))
    print("  low : " + ", ".join(f"{f}({v:+.2f})" for f, v in low.items()))

    # 主なアーティスト
    top_art = d_k['artist_name'].value_counts().head(3)
    print("  top artists: " + ", ".join(f"{a} ({n})" for a, n in top_art.items()))

    print("  --- representative tracks (closest to cluster center) ---")
    for _, row in d_k.nsmallest(N_SHOW, 'dist_to_center').iterrows():
        print_track(row)

    print("  --- most popular tracks ---")
    for _, row in d_k.nlargest(N_SHOW, 'popularity').iterrows():
        print_track(row)

print("=" * 110)

# 全曲の一覧をCSVに保存（Excelで文字化けしないよう utf-8-sig）
out_cols = ['cluster_v3', 'track_name', 'artist_name', 'popularity', 'dist_to_center', 'spotify_url'] + FEATURE_COLS_V2
csv_name = f"clusters_{TARGET_GENRE}_{TARGET_YEAR}_tracks.csv"
df_tracks.sort_values(['cluster_v3', 'dist_to_center'])[out_cols].to_csv(csv_name, index=False, encoding='utf-8-sig')
print("saved:", csv_name)

#Step6: 係数分析（Ridge回帰）

**打ち合わせメモ**: 係数分析

gb2（決定木のモデル）には係数がないので、係数分析には**線形回帰（Ridge回帰）**を使う。

- 学習 = 2020年、評価 = 2021年（gb2 と同じ条件）
- 特徴量は標準化してから学習する。そのため係数は「**その特徴が1標準偏差増えたとき、人気度が何ポイント変わるか**」と読める。特徴量どうしで大きさを比べられる
- 係数の信頼性を見るため、学習データを1000回リサンプリング（ブートストラップ）して **95%信頼区間**を出す。区間が0をまたがない係数だけを「有意」とする
- gb2 の SHAP 重要度（平均|SHAP|）と並べて、2つのモデルで重要とされる特徴が一致するかを確認する

 有意な係数があっても、R² が小さければ人気度全体への影響は小さい。
「その特徴が人気度を決めている」ではなく「わずかに傾向がある」

In [ ]:
# Step6: 係数分析（Ridge回帰 + ブートストラップ95%信頼区間）
# 必要な変数: X2_train, y_train, X2_test, y_test, gb2_pred, shap_matrix_v3, FEATURE_COLS_V2, TARGET_GENRE

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

RIDGE_ALPHA = 10.0
N_BOOT = 1000

ridge = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
ridge.fit(X2_train, y_train)
ridge_pred = ridge.predict(X2_test)

print(f"Step6: coefficient analysis ({TARGET_GENRE}, train {TARGET_YEAR} -> test {TEST_YEAR})")
print("=" * 70)
print(f"  ridge: MAE={mean_absolute_error(y_test, ridge_pred):.4f}, R2={r2_score(y_test, ridge_pred):.4f}")
print(f"  gb2  : MAE={mean_absolute_error(y_test, gb2_pred):.4f}, R2={r2_score(y_test, gb2_pred):.4f}")

coef = pd.Series(ridge.named_steps['ridge'].coef_, index=FEATURE_COLS_V2)

# 学習データをリサンプリングして係数のばらつきを見る
rng_c = np.random.default_rng(0)
Xv = X2_train.values
yv = y_train.values
boot = np.zeros((N_BOOT, len(FEATURE_COLS_V2)))
for b in range(N_BOOT):
    bi = rng_c.integers(0, len(yv), len(yv))
    m = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
    m.fit(Xv[bi], yv[bi])
    boot[b] = m.named_steps['ridge'].coef_

coef_table = pd.DataFrame({
    'coef': coef,
    'ci_low': np.percentile(boot, 2.5, axis=0),
    'ci_high': np.percentile(boot, 97.5, axis=0),
    'mean_abs_shap_gb2': np.abs(shap_matrix_v3).mean(axis=0),
})
coef_table['significant'] = (coef_table['ci_low'] > 0) | (coef_table['ci_high'] < 0)
coef_table = coef_table.reindex(coef_table['coef'].abs().sort_values(ascending=False).index)

print()
print("coef = popularity change per +1 SD of the feature (sorted by |coef|)")
print(coef_table.round(3).to_string())

# 図: 左 = Ridge係数と95%信頼区間（赤 = 有意）、右 = gb2 の SHAP 重要度
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t = coef_table.iloc[::-1]
bar_colors = ['tab:red' if s else 'lightgray' for s in t['significant']]
err_low = np.clip((t['coef'] - t['ci_low']).values, 0, None)
err_high = np.clip((t['ci_high'] - t['coef']).values, 0, None)
axes[0].barh(t.index, t['coef'], xerr=[err_low, err_high], color=bar_colors, capsize=3)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title(f"Ridge coefficients ({TARGET_GENRE}, 95% CI, red = significant)")
axes[0].set_xlabel("popularity change per +1 SD")
axes[0].grid(axis='x', alpha=0.3)

t2 = coef_table.sort_values('mean_abs_shap_gb2')
axes[1].barh(t2.index, t2['mean_abs_shap_gb2'], color='steelblue')
axes[1].set_title(f"gb2 mean |SHAP| ({TARGET_GENRE} {TARGET_YEAR})")
axes[1].set_xlabel("mean |SHAP|")
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'step6_coef_{TARGET_GENRE}.png', dpi=150, bbox_inches='tight')
plt.show()

sig = coef_table[coef_table['significant']]
print()
print("significant features:", ", ".join(f"{f}({v:+.2f})" for f, v in sig['coef'].items()) if len(sig) > 0 else "none")

#Step7: 知っている楽曲を探す

**打ち合わせメモ**: 次の進め方の指標として、自分の知っている楽曲を探す

有名アーティストの名前でデータを検索し、**見つかった曲がどのクラスタに入っているか**を表示する。

- `KNOWN_ARTISTS` に知っているアーティストを足して再実行してよい（洋楽・K-POP どちらも入れてある）
- `SEARCH_WORD` に文字を入れると、曲名とアーティスト名の部分一致で検索できる（例: `"Drake"`、`"Dynamite"`）
- クラスタごとに「知っている曲が何曲あるか」も出す。知っている曲が1曲もないクラスタは、聴いて確かめるのが難しい

`TARGET_GENRE` を `k-pop` に変えて上から実行し直せば、K-POP でも同じことができる。

In [ ]:
# Step7: 知っている楽曲を探す
# 必要な変数: df_tracks（Step5で作成。無ければ df_2020）, CHOSEN_K_V3

import re

# 知っているアーティスト（自由に足してよい）
KNOWN_ARTISTS = [
    # 洋楽ヒップホップ・ポップ
    'Drake', 'Eminem', 'Travis Scott', 'Post Malone', 'Kanye West', 'Nicki Minaj',
    'Cardi B', 'Megan Thee Stallion', 'DaBaby', 'Lil Nas X', 'Doja Cat', 'Juice WRLD',
    'XXXTENTACION', 'Machine Gun Kelly', 'The Kid LAROI', 'Jack Harlow', '21 Savage',
    'Lil Baby', 'Future', 'Playboi Carti', 'Tyler, The Creator', 'Kendrick Lamar',
    'J. Cole', 'Pop Smoke', 'Roddy Ricch', 'Bad Bunny', 'Daddy Yankee', 'Justin Bieber',
    'The Weeknd', 'Ariana Grande', 'Dua Lipa', 'Billie Eilish', 'Bruno Mars',
    # K-POP
    'BTS', 'BLACKPINK', 'TWICE', 'Stray Kids', 'SEVENTEEN', 'NCT', 'TXT',
    'TOMORROW X TOGETHER', 'ITZY', 'IU', 'EXO', 'Red Velvet', 'aespa', 'ENHYPEN',
    'MONSTA X', 'GOT7', 'NiziU', 'IVE', 'LE SSERAFIM', 'NewJeans',
]

SEARCH_WORD = "Dynamite"  # 例: "Drake" / "Dynamite" / "Butter"

# Step5 を実行済みなら df_tracks を使う。未実行なら df_2020 から作る
try:
    base = df_tracks.copy()
except NameError:
    base = df_2020.copy()
    base['spotify_url'] = 'https://open.spotify.com/track/' + base['track_id'].astype(str)


def show_track(row, show_cluster=True):
    title = str(row['track_name'])[:38].ljust(40)
    artist = str(row['artist_name'])[:22].ljust(24)
    pop = str(int(row['popularity'])).rjust(3)
    cl = f"  C{int(row['cluster_v3'])}" if show_cluster else ""
    print("    " + title + artist + pop + cl + "  " + row['spotify_url'])


pattern = "|".join(re.escape(a) for a in KNOWN_ARTISTS)
hits = base[base['artist_name'].str.contains(pattern, case=False, na=False)].copy()

print(f"Step7: known tracks in {TARGET_GENRE} {TARGET_YEAR}")
print("=" * 100)
print(f"total tracks: {len(base)} / matched by KNOWN_ARTISTS: {len(hits)} ({len(hits)/len(base)*100:.1f}%)")

print()
print("--- most popular known tracks (top 20) ---")
for _, row in hits.nlargest(20, 'popularity').iterrows():
    show_track(row)

print()
print("--- known tracks by cluster ---")
for k in range(CHOSEN_K_V3):
    size_k = int((base['cluster_v3'] == k).sum())
    hk = hits[hits['cluster_v3'] == k].sort_values('popularity', ascending=False)
    print("=" * 100)
    print(f"cluster {k}: {len(hk)} known tracks / cluster size {size_k}")
    if len(hk) == 0:
        print("    (no known tracks)")
    for _, row in hk.head(8).iterrows():
        show_track(row, show_cluster=False)

# 任意のキーワード検索
if SEARCH_WORD:
    m_title = base['track_name'].str.contains(SEARCH_WORD, case=False, na=False)
    m_artist = base['artist_name'].str.contains(SEARCH_WORD, case=False, na=False)
    found = base[m_title | m_artist].sort_values('popularity', ascending=False)
    print()
    print("=" * 100)
    print(f"search '{SEARCH_WORD}': {len(found)} tracks")
    for _, row in found.head(20).iterrows():
        show_track(row)

#Step8: 知っている3曲の特徴量とグラフ

2020年前後に流行し、洋楽に詳しくなくても知られている3曲をデータから取り出して比較する。

| 曲 | アーティスト | ジャンル | 年 | 人気度 |
|----|------------|---------|----|-------|
| Blinding Lights | The Weeknd | pop | 2020 | 90 |
| Dance Monkey | Tones And I | pop | 2019 | 76 |
| Dynamite | BTS | k-pop | 2020 | 81 |

**グラフの読み方**: 縦軸・各軸の値は **z スコア**（その曲と同じジャンル・同じ年の平均を 0、ばらつき1つ分を 1 とした値）。
0 より大きければ「同ジャンル・同年の平均より高い」、小さければ「低い」。
ジャンルが違う3曲を同じ土俵で比べるため、生の値ではなく各ジャンル内での位置に直している。


In [ ]:
# Step8: 知っている3曲の特徴量とグラフ
# 必要な変数: df_clean, FEATURE_COLS_V2（cell-2〜8, cell-10）

KNOWN_TRACKS = [
    ('0VjIjW4GlUZAMYd2vXMi3b', 'Blinding Lights'),
    ('2XU0oxnq2qxCpomAAuJY8K', 'Dance Monkey'),
    ('5QDLhrAOJJdNAmCTJ8xMyW', 'Dynamite'),
]

items = []
for tid, label in KNOWN_TRACKS:
    hit = df_clean[df_clean['track_id'] == tid]
    if len(hit) == 0:
        print("not found:", label)
        continue
    row = hit.iloc[0]
    # 同じジャンル・同じ年の曲を基準にする
    base = df_clean[(df_clean['genre'] == row['genre']) & (df_clean['year'] == row['year'])]
    z = (row[FEATURE_COLS_V2].astype(float) - base[FEATURE_COLS_V2].mean()) / base[FEATURE_COLS_V2].std()
    items.append({'label': label, 'row': row, 'z': z, 'n': len(base)})
    print(f"{label}: {row['artist_name']} / {row['genre']} {row['year']} / popularity={row['popularity']} / baseline n={len(base)}")

# 表1: 生の特徴量
raw_table = pd.DataFrame({it['label']: it['row'][FEATURE_COLS_V2].astype(float) for it in items})
print()
print("=== raw feature values ===")
print(raw_table.round(3).to_string())

# 表2: z スコア（同ジャンル・同年の平均との差）
z_table = pd.DataFrame({it['label']: it['z'] for it in items})
print()
print("=== z-score (0 = average of same genre & year) ===")
print(z_table.round(2).to_string())

# グラフ: 左=レーダーチャート / 右=横棒グラフ
labels = FEATURE_COLS_V2
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
colors = ['tab:red', 'tab:blue', 'tab:green']

fig = plt.figure(figsize=(17, 7))

ax1 = fig.add_subplot(1, 2, 1, polar=True)
for i, it in enumerate(items):
    vals = it['z'].tolist()
    vals += vals[:1]
    ax1.plot(angles, vals, 'o-', linewidth=2, color=colors[i], label=it['label'])
    ax1.fill(angles, vals, alpha=0.08, color=colors[i])
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(labels, fontsize=8)
ax1.set_ylim(-2.5, 2.5)
ax1.set_title("3曲の特徴量（同ジャンル・同年の平均を0とした値）", fontsize=12, fontweight='bold', pad=20)
ax1.legend(loc='upper right', bbox_to_anchor=(1.25, 1.12), fontsize=9)
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2)
y = np.arange(len(labels))
h = 0.26
for i, it in enumerate(items):
    ax2.barh(y + (i - 1) * h, it['z'].values, height=h, color=colors[i], label=it['label'])
ax2.set_yticks(y)
ax2.set_yticklabels(labels, fontsize=9)
ax2.axvline(0, color='black', linewidth=0.9)
ax2.set_xlabel("z スコア（0 = 同ジャンル・同年の平均）")
ax2.set_title("特徴量ごとの比較", fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('step8_known_tracks.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved: step8_known_tracks.png")


#Step9: Dynamite（原曲＋リミックス9種）がどのクラスタに入ったか

**目的**: 同じ曲・同じメロディで、アレンジだけが違う10曲がどのクラスタに入ったかを確かめる。

- **別々のクラスタに分かれる** → クラスタは「音の作り（アレンジ）」の違いを捉えている
- **全部同じクラスタ** → クラスタは音の違いをあまり捉えていない

特徴量は「k-pop 2020年の平均を0、ばらつき1つ分を1とした値（z スコア）」で表示する。
原曲と Acoustic Remix・EDM Remix を聴き比べたときの印象と、この表の値が合っているかも確認する（次回の「聴感と特徴量の比較」につなげる）。


In [ ]:
# Step9: Dynamite（原曲＋リミックス9種）のクラスタと特徴量
# 必要な変数: df_2020（cluster_v3 付き）, FEATURE_COLS_V2

SHOW_FEATS = ['danceability', 'energy', 'valence', 'acousticness', 'loudness', 'tempo', 'speechiness']

is_dyn = df_2020['track_name'].str.contains('Dynamite', case=False, na=False) & (df_2020['artist_name'] == 'BTS')
dyn = df_2020[is_dyn].copy()

# k-pop 2020年の平均を0とした z スコア
feat_mean = df_2020[FEATURE_COLS_V2].mean()
feat_std = df_2020[FEATURE_COLS_V2].std()
for f in SHOW_FEATS:
    dyn[f + '_z'] = (dyn[f] - feat_mean[f]) / feat_std[f]

dyn['version'] = dyn['track_name'].str.replace('Dynamite - ', '', regex=False).str.replace('Dynamite', 'Original', regex=False)
dyn = dyn.sort_values('popularity', ascending=False)

print(f"Dynamite versions: {len(dyn)} tracks / clusters used: {sorted(dyn['cluster_v3'].unique().tolist())}")
print()

head = "version".ljust(20) + "pop".rjust(5) + "  cl" + "".join(f[:6].rjust(8) for f in SHOW_FEATS)
print(head)
print("-" * len(head))
for _, r in dyn.iterrows():
    line = str(r['version'])[:18].ljust(20) + str(int(r['popularity'])).rjust(5) + ("C" + str(int(r['cluster_v3']))).rjust(4)
    line += "".join(f"{r[f + '_z']:+.2f}".rjust(8) for f in SHOW_FEATS)
    print(line)
print("values are z-scores (0 = k-pop 2020 average)")

# クラスタごとの内訳
print()
print("cluster breakdown:")
for cl, grp in dyn.groupby('cluster_v3'):
    print(f"  C{int(cl)}: {len(grp)} versions -> " + ", ".join(grp['version'].tolist()))

# 図: バージョンごとの特徴量（z スコア）。行ラベルの括弧内はクラスタ番号
plot_df = dyn.set_index(dyn['version'] + ' (C' + dyn['cluster_v3'].astype(int).astype(str) + ')')[[f + '_z' for f in SHOW_FEATS]]
plot_df.columns = SHOW_FEATS
fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(plot_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=ax)
ax.set_title("Dynamite の各バージョンの特徴量（k-pop 2020年平均を0、括弧内はクラスタ）", fontsize=12, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('step9_dynamite_versions.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved: step9_dynamite_versions.png")


#Step10: Dynamite の赤と青の図（force plot）

**打ち合わせメモ**: 他のやつ（Dynamite）などでも shap.initjs の赤と青の図を作ってみる

- **赤** = 人気度の予測を押し上げた特徴、**青** = 押し下げた特徴。棒が長いほど影響が大きい
- 左端の **base value** は k-pop 2020年の平均的な予測値。そこに各特徴の影響が足し引きされて、その曲の予測値（太字）になる
- 原曲と、性格の違うリミックス（Acoustic・EDM・Slow Jam）を並べ、**同じ曲でもアレンジで特徴の効き方が変わるか**を見る

**注意**: k-pop はモデルの予測力がほぼ0（Step C3 で Ridge R² −0.001）。
この図は「人気が出た理由」ではなく、**「モデルがこの曲をどう読んだか」**を表す。

上の cell-21 と同じ図だが、ここでは画像（PNG）としても保存できるよう matplotlib 版で描く。


In [ ]:
# Step10: Dynamite の force plot（赤と青の図）
# 必要な変数: df_2020（cluster_v3 付き）, shap_values_v3（cell-23 で2020年の曲に対して計算したもの）, explainer_gb2, FEATURE_COLS_V2

# shap_values_v3 は cell-23 で2020年の曲（df_2020 と同じ並び）に対して計算し直されている
assert len(shap_values_v3.values) == len(df_2020), "cell-23 を実行してからこのセルを実行してください"

TARGET_VERSIONS = ['Dynamite', 'Dynamite - Acoustic Remix', 'Dynamite - EDM Remix', 'Dynamite - Slow Jam Remix']

base_val = float(np.ravel(explainer_gb2.expected_value)[0])
pos_all = np.arange(len(df_2020))

for name in TARGET_VERSIONS:
    m = ((df_2020['track_name'] == name) & (df_2020['artist_name'] == 'BTS')).values
    if m.sum() == 0:
        print("not found:", name)
        continue
    i = int(pos_all[m][0])
    row = df_2020.iloc[i]
    pred = base_val + shap_values_v3.values[i].sum()
    print("=" * 90)
    print(f"{name} / popularity={int(row['popularity'])} / predicted={pred:.1f} / cluster C{int(row['cluster_v3'])}")
    shap.force_plot(base_val, shap_values_v3.values[i], row[FEATURE_COLS_V2].astype(float).round(3),
                    matplotlib=True, show=False, text_rotation=15)
    fname = 'step10_force_' + name.replace('Dynamite - ', '').replace('Dynamite', 'Original').replace(' ', '_') + '.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print("saved:", fname)
